# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading, overview, extraction, processing, and visualization of the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"\n{metadata['name']}: {metadata['description']}\n")
print(f"Dataset identifier: {metadata['identifier']}")
print(f"Dataset published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset metadata provides record set information. Let's enumerate available record sets, fields, and columns, referencing each by `@id`.

In [ ]:
# Find available record sets (@id)

record_sets = []

metadata_json = dataset.metadata.to_json()

# Extract record sets from metadata
if 'recordSet' in metadata_json and metadata_json['recordSet']:
    for rs in metadata_json['recordSet']:
        if isinstance(rs, dict):
            record_sets.append(rs['@id'])
        elif isinstance(rs, str):
            record_sets.append(rs)

    print(f"Record sets found: {record_sets}")
else:
    print("No record sets found directly in metadata; attempting to enumerate via dataset API.")

# If metadata['recordSet'] is empty, enumerate via dataset API
if not record_sets:
    # Try to enumerate from dataset.records()
    if hasattr(dataset, 'record_sets'):
        record_sets = [rs['@id'] for rs in dataset.record_sets()]
        print(f"Record sets found via API: {record_sets}")
    else:
        print("No record sets available.")

# Show example records for each record set
for rs_id in record_sets:
    print(f"\n--- Records from record set @id: {rs_id} ---")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            if i < 2:
                print(record)
            else:
                break
    except Exception as e:
        print(f"Unable to load records from {rs_id}: {e}")

# If no record sets found, print sample access
if not record_sets:
    print("Check dataset documentation or schema for valid record set IDs.")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.

Below, all record sets found in the overview are extracted. Columns are referenced by their `@id`.

In [ ]:
# Extract data from each record set (@id)
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame columns for record set {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for record set {record_set_id}")

# Pick a record set for downstream analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nUsing record set @id: {main_record_set_id} for EDA.")
else:
    print("No dataframes available for EDA.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records by criteria, normalizing numeric fields, and categorizing data.

**Note:** All fields/columns are referenced by their `@id`.

This section demonstrates removing outliers, transforming distributions, and grouping by key attributes.

In [ ]:
# EDA using available DataFrame and column @ids
# Display columns and pick numeric/grouping fields
df = main_df
columns = df.columns.tolist()
print(f"Available columns (@id): {columns}")

# Try to pick a numeric field (e.g., age) via @id
numeric_field_id = None
possible_numeric_ids = [col for col in columns if 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower()]
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
else:
    # fallback: use the first column
    numeric_field_id = columns[0]

print(f"Selected numeric field for analysis (@id): {numeric_field_id}")

# Select threshold for filtering
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold]
else:
    # Convert to numeric if possible
    temp_col = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[temp_col > threshold].copy()
    filtered_df[numeric_field_id] = temp_col[temp_col > threshold]

print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if len(filtered_df) > 0 and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

# Try to pick a grouping field (e.g., sex, location, status)
group_field_id = None
possible_group_ids = [col for col in columns if 'sex' in col.lower() or 'location' in col.lower() or 'status' in col.lower() or 'anatomical' in col.lower()]
if possible_group_ids:
    group_field_id = possible_group_ids[0]
print(f"Grouping field selected (@id): {group_field_id}")

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This example creates a histogram of the numeric field and a barplot for grouping by a categorical attribute (referenced by their `@id`).

In [ ]:
# Visualize histogram for selected numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Barplot for grouping field
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, process, and visualize a FAIR²-compliant clinical dataset with `mlcroissant`.

- All entities were referenced by their `@id` fields (record sets, fields, columns).
- Example EDA and visualizations highlighted possible clinical patterns in second primary colorectal cancer survivors.
- Data can now be further used for statistical or machine learning analysis.

For reproducible provenance, all identifiers and operations follow the Croissant schema specification.